# Support Vector Machines for Nutrition Research

**Author:** [Author Name]  
**Date:** [Date]

---

## Introduction

Support vector machines (SVMs) are a supervised learning method that finds the hyperplane best separating two classes in a high-dimensional feature space. Rather than fitting a model to all training observations, the SVM identifies a small subset — the **support vectors** — that lie closest to the decision boundary, and it maximizes the margin between those points and the boundary. This combination of geometric clarity and strong regularization makes SVMs effective when predictors are numerous relative to sample size, a situation common in nutrition epidemiology.

In this tutorial, we apply SVM to predict **extreme blood pressure** — defined as systolic ≥ 130 mmHg or diastolic ≥ 80 mmHg (AHA/ACC 2017 Stage 1 hypertension) — using dietary, anthropometric, and laboratory data from the NHANES 1999–2018 dataset. We walk through feature selection, data preparation, model training, and performance evaluation, ending with a visualization of the SVM decision boundary projected onto two dimensions. The goal is not to build the best possible predictive model, but to illustrate how SVMs work and how to interpret their output in a nutrition research context.

---

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # Suppress known sklearn convergence warnings

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.svm import SVC                          # SVM classifier
from sklearn.ensemble import RandomForestClassifier   # Random forest (feature selection)
from sklearn.feature_selection import RFECV           # Recursive feature elimination with CV
from sklearn.model_selection import train_test_split  # Data splitting
from sklearn.model_selection import StratifiedKFold   # Stratified cross-validation folds
from sklearn.preprocessing import StandardScaler      # Feature standardization
from sklearn.metrics import classification_report     # Performance metrics
from sklearn.metrics import ConfusionMatrixDisplay    # Confusion matrix visualization

# ---- Shared constants ----
SEED = 42
np.random.seed(SEED)

# ---- Plot style (call before every figure in this tutorial) ----
def set_plot_style():
    sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.2)
    plt.rcParams.update({
        'figure.figsize': (7, 5),
        'axes.titlesize': 14,
        'axes.labelsize': 12,
        'figure.dpi': 100,
    })

set_plot_style()

---

## Data

The NHANES dataset used here contains 21,849 adults with complete fasting laboratory and dietary recall data across ten two-year survey cycles (1999–2018). See the [data README](../../data/README.md) for a full variable dictionary.

Two variables are excluded before modeling:

- **`bp_sys` and `bp_di`** are excluded because they directly define the outcome (`extreme_bp = bp_sys >= 130 | bp_di >= 80`). Including them would be data leakage.
- **`metsyn`** is excluded because extreme blood pressure is one of its five components. A positive `metsyn` mechanically increases the probability of `extreme_bp = True`, which would inflate model performance.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/m-powell/MLforNutrition/master/data/MLforNutrition_NHANES.csv'

try:
    nhanes = pd.read_csv(DATA_URL)
    print(f'Loaded from URL: {nhanes.shape[0]:,} rows, {nhanes.shape[1]} columns')
except Exception:
    print('Could not load from URL; falling back to local copy.')
    nhanes = pd.read_csv('../../data/MLforNutrition_NHANES.csv')

nhanes.head()

We encode `extreme_bp` as an integer (0/1) and `gender` as a binary indicator (1 = male). The `dropna()` call is a safeguard; all variables used for modeling are complete in this dataset after the upstream pipeline.

In [ ]:
# Predictor set for feature selection
# Excludes bp_sys/bp_di (define outcome), metsyn (contains outcome),
# meds_hbp/meds_chol (many NAs, not dietary predictors),
# identifiers, and imputation flags
feature_vars = [
    'age', 'gender',
    'weight', 'height', 'bmi', 'waist',
    'kcal', 'protein', 'sugar', 'carb',
    'fat_total', 'fat_sat', 'fat_mon', 'fat_poly',
    'hdl', 'glucose', 'triglycerides'
]

nhanes_model = (
    nhanes
    .dropna(subset=feature_vars + ['extreme_bp'])
    .assign(
        extreme_bp=lambda df: df['extreme_bp'].astype(int),
        gender_bin=lambda df: (df['gender'] == 'male').astype(int)
    )
    .reset_index(drop=True)
)

# Replace 'gender' with the encoded binary column for sklearn
feature_vars_enc = ['gender_bin' if v == 'gender' else v for v in feature_vars]

n_obs = len(nhanes_model)
prev  = nhanes_model['extreme_bp'].mean() * 100
print(f'Observations for modeling: {n_obs:,}')
print(f'Extreme BP prevalence:     {prev:.1f}%')

---

## Method

### Feature Selection

Before fitting the SVM, we rank predictor importance using a random forest. Random forests build many decision trees on random subsets of predictors, and each variable's importance is measured by how much it reduces classification impurity (the Gini index) across all trees. We then refine the selection using **recursive feature elimination with cross-validation (RFECV)**, which fits models while incrementally removing the least important feature and retains the subset with the best cross-validated accuracy.

Note: the RFECV step uses 5-fold cross-validation and may take several minutes to run.

In [ ]:
X_fs = nhanes_model[feature_vars_enc]
y_fs = nhanes_model['extreme_bp']

rf = RandomForestClassifier(n_estimators=200, random_state=SEED)
rf.fit(X_fs, y_fs)

# Human-readable labels (replace gender_bin -> gender for display)
labels = [v.replace('gender_bin', 'gender') for v in feature_vars_enc]
importances = pd.Series(rf.feature_importances_, index=labels).sort_values()

set_plot_style()
fig, ax = plt.subplots()
importances.plot.barh(ax=ax, color=sns.color_palette('Set2')[0])
ax.set_xlabel('Mean decrease in Gini impurity')
ax.set_title('Random Forest Variable Importance')
plt.tight_layout()
plt.show()

# Figure 1: Variable importance from a random forest classifier predicting extreme blood pressure.
# Variables are ranked by mean decrease in Gini impurity across all trees.

We next use RFECV to identify the smallest subset of features that retains the highest cross-validated accuracy. RFECV wraps any sklearn estimator and removes the least important feature at each step, keeping the configuration that maximizes cross-validated performance. In Python, this replaces the `caret::rfe()` call used in the [R companion tutorial](svm-r.html).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rfecv = RFECV(
    estimator=RandomForestClassifier(n_estimators=100, random_state=SEED),
    step=1,
    cv=cv,
    scoring='accuracy',
    min_features_to_select=1,
    n_jobs=-1
)
rfecv.fit(X_fs, y_fs)

selected = [v.replace('gender_bin', 'gender')
            for v, s in zip(feature_vars_enc, rfecv.support_) if s]
n_opt = rfecv.n_features_
print(f'Optimal number of features: {n_opt}')
print('Selected features:', selected)

mean_scores = rfecv.cv_results_['mean_test_score']

set_plot_style()
fig, ax = plt.subplots()
ax.plot(range(1, len(mean_scores) + 1), mean_scores,
        marker='o', markersize=5, color=sns.color_palette('Set2')[0])
ax.axvline(n_opt, linestyle='--', color='gray', linewidth=1,
           label=f'Optimal ({n_opt} features)')
ax.set_xlabel('Number of features')
ax.set_ylabel('Cross-validated accuracy')
ax.set_title('RFECV: Accuracy vs. Number of Features')
ax.legend()
plt.tight_layout()
plt.show()

# Figure 2: Cross-validated accuracy as a function of the number of features retained by RFECV.
# The dashed line marks the optimal subset size.

Both methods consistently identify anthropometric predictors — particularly waist circumference, height, weight — and metabolic markers (triglycerides, HDL, glucose) as the strongest predictors. Based on these results and subject-matter judgment, we fit the SVM using seven predictors: `height`, `weight`, `triglycerides`, `waist`, `protein`, `age`, and `glucose`.

### Data Splitting

We split the dataset into 80% training and 20% testing, stratifying on the outcome to maintain class balance in both sets. In Python, `train_test_split(..., stratify=y)` achieves the same stratified split as `caret::createDataPartition()` in R.

In [ ]:
model_vars = ['height', 'weight', 'triglycerides', 'waist', 'protein', 'age', 'glucose']

X = nhanes_model[model_vars]
y = nhanes_model['extreme_bp']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f'Training set: {len(X_train):,} observations')
print(f'Test set:     {len(X_test):,} observations')

### Standardization

SVMs rely on distances between observations in feature space. Variables measured on different scales — waist circumference in centimeters versus protein intake in grams per day — will dominate the distance calculation by magnitude alone, distorting the decision boundary. We therefore standardize all predictors to mean 0 and standard deviation 1.

Crucially, scaling parameters (mean and SD) are computed from the **training set only** and then applied to both sets. In scikit-learn, `StandardScaler.fit_transform()` computes and applies these parameters on the training set, while `transform()` applies the already-fitted parameters to the test set without refitting.

In [ ]:
scaler = StandardScaler()

X_train_sc = pd.DataFrame(
    scaler.fit_transform(X_train),  # Fit on train, transform train
    columns=model_vars,
    index=X_train.index             # Preserve indices for label alignment
)
X_test_sc = pd.DataFrame(
    scaler.transform(X_test),       # Transform test with train parameters only
    columns=model_vars,
    index=X_test.index
)

### Model Setup and Hyperparameters

The SVM has two key settings for a linear kernel:

- **Kernel:** We use a **linear kernel**, which fits a flat hyperplane as the decision boundary. This is appropriate here because we want an interpretable model and because the selected predictors have a reasonably linear relationship with the outcome. Non-linear kernels (radial basis function, polynomial) can capture more complex boundaries but are harder to interpret.

- **Cost (C):** The cost parameter controls the trade-off between margin width and training errors. A **larger C** penalizes misclassifications more heavily, producing a narrow margin that fits training data closely (lower bias, higher variance). A **smaller C** allows more training errors in exchange for a wider, more generalizable margin (higher bias, lower variance). We use `C = 10` here; in practice, C should be tuned by cross-validation.

In scikit-learn, SVM classification is handled by `SVC`. Setting `kernel='linear'` and `C=10` matches the linear SVM fit in the [R companion tutorial](svm-r.html).

### Fitting the Model

In [ ]:
svm_model = SVC(kernel='linear', C=10, random_state=SEED)
svm_model.fit(X_train_sc, y_train)

n_sv = sum(svm_model.n_support_)
print(svm_model)
print(f'\nNumber of support vectors: {n_sv:,}')

### Evaluating Performance

We predict outcomes for the held-out test set and summarize performance with a classification report and confusion matrix. Beyond accuracy, we pay attention to **recall** (the proportion of true extreme-BP cases correctly identified — equivalent to sensitivity in the R output) and **precision** (the proportion of predicted positives that are true positives), since the cost of a missed positive case may differ from the cost of a false alarm in a clinical screening context.

In [ ]:
y_pred = svm_model.predict(X_test_sc)

print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Extreme (1)']))

set_plot_style()
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Normal', 'Extreme'],
    colorbar=False,
    ax=ax,
    cmap='Blues'
)
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.show()

# Figure 3: Confusion matrix for the linear SVM on the held-out test set.
# Rows are true classes; columns are predicted classes.

### Decision Boundary Visualization

To plot the decision boundary, we fit a second SVM using only `height` and `weight`. This collapses the feature space to two dimensions so the boundary can be drawn directly. The two-variable model is for illustration only — its accuracy will be lower than the full model.

With a linear kernel, scikit-learn exposes the weight vector directly via `coef_` and the bias via `intercept_`. The decision boundary satisfies `w[0]*height + w[1]*weight + b = 0`, which we solve for `weight` to get the slope and intercept of the boundary line.

In [ ]:
X_train_2v = X_train_sc[['height', 'weight']]
X_test_2v  = X_test_sc[['height', 'weight']]

svm_2v = SVC(kernel='linear', C=10, random_state=SEED)
svm_2v.fit(X_train_2v, y_train)

y_pred_2v = svm_2v.predict(X_test_2v)
acc_2v    = (y_pred_2v == y_test.values).mean() * 100
print(f'Two-variable model accuracy: {acc_2v:.1f}%')

In [ ]:
w = svm_2v.coef_[0]       # Weight vector [w_height, w_weight]
b = svm_2v.intercept_[0]  # Bias term

# Decision boundary: w[0]*height + w[1]*weight + b = 0
# Solved for weight:  weight = -(w[0]*height + b) / w[1]
slope_db     = -w[0] / w[1]
intercept_db = -b   / w[1]
margin_off   =  1.0 / w[1]  # Margin lines are at w.x + b = ±1

X_sv = X_train_2v.iloc[svm_2v.support_]  # Support vectors (positional)

h_range = np.linspace(X_train_2v['height'].min() - 0.5,
                      X_train_2v['height'].max() + 0.5, 300)

palette = {0: sns.color_palette('Set2')[0], 1: sns.color_palette('Set2')[1]}

set_plot_style()
fig, ax = plt.subplots()

for cls, label in [(0, 'No'), (1, 'Yes')]:
    mask = (y_train == cls)
    ax.scatter(
        X_train_2v.loc[mask, 'height'],
        X_train_2v.loc[mask, 'weight'],
        color=palette[cls], alpha=0.25, s=8, label=label
    )

ax.scatter(
    X_sv['height'], X_sv['weight'],
    facecolors='none', edgecolors='#404040', s=60, linewidths=0.8
)

ax.plot(h_range, slope_db * h_range + intercept_db,
        color='black', linewidth=1.2)
ax.plot(h_range, slope_db * h_range + intercept_db + margin_off,
        color='black', linewidth=0.6, linestyle='--')
ax.plot(h_range, slope_db * h_range + intercept_db - margin_off,
        color='black', linewidth=0.6, linestyle='--')

ax.set_xlabel('Height (standardized)')
ax.set_ylabel('Weight (standardized)')
ax.set_title('SVM Decision Boundary', fontsize=14, fontweight='bold')
ax.legend(title='Extreme BP', loc='lower right')
plt.tight_layout()
plt.show()

# Figure 4: Linear SVM decision boundary (solid line) with margins (dashed lines) for a
# two-variable model predicting extreme blood pressure using standardized height and weight.
# Open circles mark support vectors. This simplified model is for visualization only;
# the full model uses 7 predictors.

---

## Results

The seven-variable SVM achieved approximately 63% accuracy on the held-out test set. While this exceeds the no-information rate, recall — the model's ability to identify individuals who truly have extreme blood pressure — was notably lower than precision, indicating the model is conservative in flagging positive cases. This reflects the moderate class imbalance in the dataset: extreme blood pressure is more common than not in adults (roughly 55% prevalence in this NHANES sample), which pushes the model toward the majority class.

Among predictors, waist circumference, height, weight, and triglyceride levels were the most informative features in both the random forest ranking and RFECV. This is consistent with the established cardiometabolic risk literature, where abdominal adiposity and dyslipidemia are recognized precursors to hypertension. The modest overall accuracy also reflects a fundamental limitation of the predictor set: dietary and anthropometric variables capture only a portion of the variance in blood pressure, which is also influenced by genetic factors, medication, stress, and other variables not available in this dataset.

---

## Summary

**Key takeaways:**

- SVMs find the maximum-margin hyperplane separating two classes and are defined by the support vectors — the small subset of training observations closest to the boundary.
- Standardizing predictors before fitting an SVM is essential because the algorithm is sensitive to variable scale. In scikit-learn, `StandardScaler` should be fit on the training set only, then applied to both sets.
- Feature selection via random forest importance and RFECV identified anthropometric predictors (waist, height, weight) and metabolic markers (triglycerides, glucose) as most informative for predicting extreme blood pressure.
- Excluding variables that define or contain the outcome (`bp_sys`, `bp_di`, `metsyn`) is critical to avoid data leakage, which would artificially inflate model performance.
- The ~63% accuracy achieved here is meaningful but modest, reflecting the complexity of blood pressure as a phenotype and the limitations of dietary and anthropometric data as sole predictors.

**Limitation:** SVMs with linear kernels assume the decision boundary is approximately flat in the feature space. When the true boundary is non-linear, a radial basis function (RBF) or polynomial kernel will perform better, though at the cost of interpretability.

**See also:** The [R companion tutorial](svm-r.html) covers the same analysis using the `e1071` package.

In [ ]:
# Session info
import sys, sklearn, matplotlib, seaborn
for mod in [sys, np, pd, sklearn, matplotlib, seaborn]:
    name = getattr(mod, '__name__', 'sys')
    ver  = getattr(mod, '__version__', sys.version.split()[0])
    print(f'{name}: {ver}')